In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# A digital twin of the SO-101, from first principles

A digital twin is only useful if code cannot tell it from the robot. That
reduces to three claims, each checked below with a measurement:

1. **It moves like the arm** - same command interface, same servo dynamics.
2. **It sees like the arm** - a camera rendered where the real one is mounted.
3. **It feels like the arm** - gripper current that comes from contact.

Then the payoff: the workshop's own `run_mimic` drives the twin, unmodified.

What is simulated: the arm, its servos, the wrist camera and the table
scene (MuJoCo). What is real: everything above the arm transport - your
webcam, YOLOv26s pose and detection, MediaPipe Hands, the mapper,
`run_mimic`. Everything runs inside this kernel; ROS 2 is not needed.

In [ ]:
import os, sys, pathlib, threading, time

os.environ.setdefault("MUJOCO_GL", "egl")  # headless rendering on the iGPU; set before mujoco loads

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

import cv2, numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events

from common.config import load_config

cfg = load_config()
cfg["robot"]["backend"] = "sim"  # the MuJoCo twin instead of the Feetech bus
cfg["robot"]["use_ros2"] = False  # in-process: this kernel owns the (simulated) bus
cfg["cameras"]["mount"]["use_ros2"] = False  # and reads the webcam itself - no ROS 2 needed
LOAD_THRESHOLD = 120  # Present_Current counts the grip behavior treats as contact

## 0. A body

The twin starts as a physics model: the SO-101's links, joints and
actuators (the open MJCF model), on a table with a few objects. MuJoCo
steps it 500 times per second on a CPU thread, locked to wall-clock time -
so one second of waiting is one second of simulated motion.

In [ ]:
from common.motion import sim_world

world = sim_world(cfg)
m = world.model
print(f"{m.nbody} bodies, {m.njnt} joints, {m.nu} actuators, timestep {world.dt * 1000:.0f} ms")
print("objects:", world.objects)

t0, w0 = world.time, time.perf_counter()
time.sleep(1.0)
print(f"sim time advanced {world.time - t0:.3f} s in {time.perf_counter() - w0:.3f} s of wall time")

## 1. It moves like the arm

`make_arm_backend(cfg)` returns `SimArmClient`, a subclass of the real
`LeRobotArm` in which only the LeRobot `SO101Follower` is swapped for a
simulated bus. Joint clamping, the goal-speed/acceleration register writes
and the current readback all run through the same code as on the rig.

Below the bus sits the part that matters physically: an STS3215 servo does
not jump to a goal. Its firmware accelerates, cruises and decelerates (a
trapezoidal profile), and the twin's servo model does the same. Command one
60° step and measure what comes back.

In [ ]:
from common.motion import LeRobotArm, make_arm_backend

arm = make_arm_backend(cfg)
print(f"{type(arm).__name__} is a LeRobotArm: {isinstance(arm, LeRobotArm)}")
print("at rest:", {k: round(v, 1) for k, v in arm.get_joints().items()})


def step_response(joint, delta, seconds=1.5, hz=100):
    """Send one goal, then sample the joint - what a scope on the servo would show."""
    q0 = arm.get_joints()[joint]
    arm.send_joints({joint: q0 + delta})
    t, q, t0 = [], [], time.perf_counter()
    while (now := time.perf_counter() - t0) < seconds:
        t.append(now)
        q.append(arm.get_joints()[joint])
        time.sleep(1.0 / hz)
    return np.array(t), np.array(q) - q0


t, q = step_response("shoulder_pan", 60.0)
v = np.gradient(q, t)
t95 = t[np.argmax(q >= 0.95 * 60.0)]
print(f"peak speed {v.max():.0f} deg/s, 95% of the step after {t95:.2f} s")

fig, ax = plt.subplots(figsize=(8, 3))
ax.step([0, 0, t[-1]], [0, 60, 60], "k--", lw=1, where="post", label="goal")
ax.plot(t, q, label="position (deg)")
ax.plot(t, v / 10, label="speed (deg/s / 10)")
ax.set_xlabel("seconds after the command")
ax.legend()
plt.show()
arm.go_to_rest()

## 2. It sees like the arm

A camera is a pinhole at a pose. The twin's `wrist_cam` sits where the real
arm camera is mounted - beside the gripper, looking along the jaws - so it
moves with the wrist and sees what is being gripped. A render thread draws
every camera on the iGPU (EGL), separate from the physics thread, and
`SimCamera.read()` returns `(ok, bgr)` exactly like `cv2.VideoCapture`.

Present the open jaws, put the cube between them, and look from both the
wrist and a free 3D view.

In [ ]:
from so101_sim import SimCamera
from so101_sim.render import shared_renderer

PRESENT = {"shoulder_pan": 0.0, "shoulder_lift": -35.0, "elbow_flex": 30.0, "wrist_flex": 10.0, "wrist_roll": 0.0}

wrist = SimCamera(world, "arm")  # the arm camera, on the wrist
view = SimCamera(world, "view", width=1280, height=720)  # a free 3D view, for us

arm.go_to({**PRESENT, "gripper": 85.0})
time.sleep(1.0)  # let the jaws finish opening
world.stage_object("cube", PRESENT)
_, w_img = wrist.read(fresh=True)
_, v_img = view.read(fresh=True)
print("renderer:", shared_renderer(world).renderer)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, img, title in zip(axes, (w_img, v_img), ("wrist camera", "3D view")):
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis("off")
plt.show()

## 3. It feels like the arm

The real gripper has no touch sensor. The grip behavior infers contact from
the servo's `Present_Current`: closing on nothing, the motor reaches its
goal and the current falls away; closing on an object, the jaws stall and
the motor pushes against it, so the current stays high.

The twin produces that current from the physics - the actuator effort the
contact actually demands - not from a script. Close the jaws twice, once
empty and once on the cube, and compare against the threshold.

In [ ]:
from common.motion import GRIPPER_CLOSED_FLOOR


def squeeze(with_cube, seconds=2.0, hz=20):
    """Open at PRESENT, (place the cube), close to the floor; return the current trace."""
    arm.go_to({**PRESENT, "gripper": 85.0})
    time.sleep(1.0)
    if with_cube:
        world.stage_object("cube", PRESENT)
    else:
        world.hide("cube")
    arm.send_joints({"gripper": GRIPPER_CLOSED_FLOOR})
    trace = []
    for _ in range(int(seconds * hz)):
        time.sleep(1.0 / hz)
        trace.append(arm.get_gripper_load() or 0.0)
    _, img = wrist.read(fresh=True)
    arm.go_to({"gripper": 85.0})  # let go
    return np.array(trace), img


empty, _ = squeeze(with_cube=False)
held, held_view = squeeze(with_cube=True)
print(f"peak current - empty jaws: {empty.max():.0f}   cube: {held.max():.0f}   (threshold {LOAD_THRESHOLD})")

fig, (ax, ax_img) = plt.subplots(1, 2, figsize=(12, 3.8))
t = np.arange(len(held)) / 20
ax.plot(t, empty, label="empty jaws")
ax.plot(t, held, label="cube between the jaws")
ax.axhline(LOAD_THRESHOLD, color="k", ls="--", lw=1, label="grip threshold")
ax.set_xlabel("seconds after close")
ax.set_ylabel("Present_Current (counts)")
ax.legend()
ax_img.imshow(cv2.cvtColor(held_view, cv2.COLOR_BGR2RGB))
ax_img.set_title("wrist camera while gripping")
ax_img.axis("off")
plt.show()
world.place("cube", world.model.body("cube").pos)  # back on the table for the next section
arm.go_to_rest()

## 4. Close the loop

If the twin moves, sees and feels like the arm, the workshop's code can drive
it without knowing. Perception is the rig's: your webcam
(`cameras.mount.device`), YOLOv26s-pose and YOLOv26s detection on the NPU
(CPU if the NPU is unavailable - the provider is printed), MediaPipe Hands on
the CPU. Without a webcam, a scripted person stands in, and the cell says so.

The loop itself is the canonical `run_mimic`, unmodified, on a thread. Like
on the rig it switches to the live envelope (the twin reports itself as a
live arm) and parks the arm when stopped. The notebook only watches:
`Tap` passes every call through and remembers the last result for display.

Top row: what YOLO-pose sees, what MediaPipe sees, and what YOLO detection
finds in the arm camera. Below: the twin, moving.

In [ ]:
from common.fixtures import ScriptedHands, ScriptedPose, SyntheticCamera
from common.motion import mapper_from_config
from integration.dispatch import WorkshopContext
from ros.camera_client import make_camera


def perception():
    """Webcam + the three models; a scripted person if the webcam gives no frames."""
    from models.hands import HandTracker
    from models.vision import ObjectDetector, PoseEstimator

    pose, hands, detector = PoseEstimator(cfg), HandTracker(cfg), ObjectDetector(cfg)
    print(f"YOLO-pose on {pose.provider}, YOLO detect on {detector.provider}, MediaPipe Hands on CPU")
    cam = make_camera(cfg, "mount")  # cameras.mount.use_ros2 is off: a direct OpenCV capture
    if cam.read()[0]:
        print(f"webcam: {cfg['cameras']['mount']['device']}")
        return cam, pose, hands, detector
    cam.close()
    print(f"webcam: no frames from {cfg['cameras']['mount']['device']} (in use by another kernel?) "
          "- a scripted person stands in")
    return SyntheticCamera(), ScriptedPose(), ScriptedHands(), detector


class Tap:
    """Pass-through that remembers the last result of one method (display only)."""

    def __init__(self, inner, method):
        self.inner, self.last = inner, None
        setattr(self, method, lambda *a, **k: self._keep(getattr(inner, method)(*a, **k)))

    def _keep(self, result):
        self.last = result
        return result

    def __getattr__(self, name):
        return getattr(self.inner, name)


cam, pose, hands, detector = perception()
ctx = WorkshopContext(
    cfg=cfg,
    arm=arm,
    camera=Tap(cam, "read"),
    pose=Tap(pose, "infer"),
    hands=Tap(hands, "process"),
    mapper=mapper_from_config(cfg),
    headless=True,
)

In [ ]:
from common.imaging import draw_skeleton
from integration.mimic import run_mimic


def jpeg(img):
    if img is None:  # a render frame that timed out - show black rather than raise
        img = np.zeros((480, 640, 3), np.uint8)
    return cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 80])[1].tobytes()


def what_the_models_saw():
    """The last webcam frame mimic consumed, drawn once with YOLO-pose and once with MediaPipe."""
    ok, frame = ctx.camera.last or (False, None)
    if not ok:
        return None, None
    yolo = frame
    if ctx.pose.last is not None and len(ctx.pose.last[0]):
        yolo = draw_skeleton(frame, *ctx.pose.last[:3], getattr(pose, "kpt_threshold", 0.3))
    hand = hands.draw(frame, *ctx.hands.last) if ctx.hands.last is not None else frame
    return yolo, hand


def what_the_arm_sees():
    ok, frame = wrist.read()
    return detector.draw(frame, detector.infer(frame)) if ok else None


def hud(img):
    load = arm.get_gripper_load() or 0.0
    color = (0, 0, 255) if load >= LOAD_THRESHOLD else (0, 200, 0)
    cv2.putText(img, f"gripper current {load:.0f} / {LOAD_THRESHOLD}", (20, 45), 0, 1.1, color, 2)
    joints = "   ".join(f"{k.split('_')[-1]} {v:.0f}" for k, v in arm.get_joints().items())
    cv2.putText(img, joints, (20, 90), 0, 0.8, (255, 255, 255), 2)
    return img


def tile(title, width):
    img = widgets.Image(format="jpeg", layout=widgets.Layout(width=width))
    return img, widgets.VBox([widgets.HTML(f"<b>{title}</b>"), img])


yolo_img, yolo_box = tile(f"YOLOv26s-pose ({getattr(pose, 'provider', 'scripted')})", "400px")
hand_img, hand_box = tile("MediaPipe Hands (CPU)", "400px")
arm_img, arm_box = tile(f"arm camera - YOLOv26s detect ({detector.provider})", "400px")
twin_img, twin_box = tile("digital twin", "1212px")
stop = widgets.Button(description="stop", button_style="danger")
stop.on_click(lambda _: ctx.stop_event.set())
display(widgets.VBox([widgets.HBox([yolo_box, hand_box, arm_box]), twin_box, stop]))

ctx.stop_event.clear()
loop = threading.Thread(target=run_mimic, args=(ctx,), daemon=True)
loop.start()
with ui_events() as poll:
    while loop.is_alive():
        poll(10)  # deliver the stop click while this cell is running
        yolo, hand = what_the_models_saw()
        yolo_img.value, hand_img.value = jpeg(yolo), jpeg(hand)
        arm_img.value = jpeg(what_the_arm_sees())
        ok, frame = view.read()
        twin_img.value = jpeg(hud(frame) if ok else None)
print("mimic stopped, arm parked")

## Clean up

On the full system, `python app.py --digital-twin` runs the same twin behind ROS 2:
the node supervisor starts `so101_sim.node` in place of the arm node and the
arm camera node, and the arm and camera clients talk to it over the usual
topics.

In [ ]:
from so101_sim import close_shared_world

ctx.stop_event.set()
loop.join(timeout=5)
arm.close()
for thing in (wrist, view, cam, hands):
    (getattr(thing, "close", None) or getattr(thing, "release", None) or (lambda: None))()
close_shared_world()
print("twin closed")